In [1]:
# ==================== CELL 1: Install & Import ====================
import pandas as pd
import numpy as np
from scipy.linalg import eig
import plotly.express as px
from google.colab import files

print("✅ Libraries ready")

✅ Libraries ready


In [5]:
# ==================== CELL 2: Load YOUR FIXED DATA ====================
# ←←← CHANGE THIS NAME ONLY IF YOUR FILE HAS A DIFFERENT NAME
df = pd.read_excel("Healthcare_TOPSIS_Cleaned_ Final.xlsx", sheet_name="Sheet1")

print("✅ FIXED Data loaded!")
print(f"Shape: {df.shape} | States: {len(df)}")
print("Disease Burden summary (should have NO zeros now):")
print(df['Disease Burden'].describe())
df.head()

✅ FIXED Data loaded!
Shape: (36, 7) | States: 36
Disease Burden summary (should have NO zeros now):
count     36.000000
mean      21.859722
std       16.076180
min        2.600000
25%       19.270000
50%       19.270000
75%       19.270000
max      100.000000
Name: Disease Burden, dtype: float64


,State,Hospital Beds,Life Expectancy,Death Rate,Disease Burden,Population Density,Poverty (Illiteracy %)
0,Andaman And Nicobar Islands,36,70.3,5.8,19.27,46.0,13.37
1,Andhra Pradesh,928,70.6,6.3,19.27,173.7,32.98
2,Arunachal Pradesh,238,70.3,5.7,19.27,17.0,34.61
3,Assam,1729,67.9,6.2,19.27,398.0,27.81
4,Bihar,3034,69.5,5.4,7.29,1106.0,38.20


In [3]:
# ==================== CELL 3: AHP Weights (same consistent matrix) ====================
criteria = ['Hospital Beds', 'Life Expectancy', 'Death Rate', 'Disease Burden',
            'Population Density', 'Poverty (Illiteracy %)']

pairwise = np.array([
    [1,     0.5,   0.333, 0.2,   0.333, 0.25],
    [2,     1,     0.5,   0.333, 0.5,   0.333],
    [3,     2,     1,     0.5,   2,     1],
    [5,     3,     2,     1,     3,     2],
    [3,     2,     0.5,   0.333, 1,     0.5],
    [4,     3,     1,     0.5,   2,     1]
])

eigenvalues, eigenvectors = eig(pairwise)
max_eig = np.max(np.real(eigenvalues))
principal = np.real(eigenvectors[:, np.argmax(np.real(eigenvalues))])
ahp_weights = principal / np.sum(principal)

# Consistency check
n = 6
CI = (max_eig - n) / (n - 1)
CR = CI / 1.24
print("✅ AHP Weights (CR =", round(CR, 4), "< 0.1 → Excellent)")
for crit, w in zip(criteria, ahp_weights):
    print(f"   {crit:25} : {w:.4f} ({w*100:.1f}%)")

✅ AHP Weights (CR = 0.0162 < 0.1 → Excellent)
   Hospital Beds             : 0.0526 (5.3%)
   Life Expectancy           : 0.0871 (8.7%)
   Death Rate                : 0.1887 (18.9%)
   Disease Burden            : 0.3339 (33.4%)
   Population Density        : 0.1261 (12.6%)
   Poverty (Illiteracy %)    : 0.2116 (21.2%)


In [6]:
# ==================== CELL 4: TOPSIS Function & Ranking ====================
def topsis(df, weights, benefit_cols, cost_cols):
    benefit = df[benefit_cols].values
    cost = df[cost_cols].values
    norm_b = benefit / np.sqrt(np.sum(benefit**2, axis=0))
    norm_c = cost / np.sqrt(np.sum(cost**2, axis=0))
    v = np.hstack((norm_b * weights[:len(benefit_cols)], norm_c * weights[len(benefit_cols):]))

    ideal_best = np.max(v[:, :2], axis=0).tolist() + np.min(v[:, 2:], axis=0).tolist()
    ideal_worst = np.min(v[:, :2], axis=0).tolist() + np.max(v[:, 2:], axis=0).tolist()

    dist_best = np.sqrt(np.sum((v - ideal_best)**2, axis=1))
    dist_worst = np.sqrt(np.sum((v - ideal_worst)**2, axis=1))
    closeness = dist_worst / (dist_best + dist_worst)

    result = df.copy()
    result['TOPSIS Score'] = closeness.round(4)
    result['Rank'] = result['TOPSIS Score'].rank(ascending=False).astype(int)
    return result.sort_values('Rank').reset_index(drop=True)

benefit_cols = ['Hospital Beds', 'Life Expectancy']
cost_cols = ['Death Rate', 'Disease Burden', 'Population Density', 'Poverty (Illiteracy %)']

ranked = topsis(df, ahp_weights, benefit_cols, cost_cols)

print("✅ TOPSIS Ranking with FIXED data is ready!")
ranked[['Rank', 'State', 'TOPSIS Score'] + criteria].head(15)

✅ TOPSIS Ranking with FIXED data is ready!


,Rank,State,TOPSIS Score,Hospital Beds,Life Expectancy,Death Rate,Disease Burden,Population Density,Poverty (Illiteracy %)
0,1,Karnataka,0.8565,10684,69.8,6.2,7.81,319.0,24.63
1,2,Maharashtra,0.8551,3203,72.9,5.5,2.60,365.0,17.66
2,3,Tamil Nadu,0.8305,2439,73.2,6.1,7.81,555.0,19.91
3,4,Mizoram,0.7895,113,70.3,4.2,19.27,52.0,8.67
4,5,Tripura,0.7809,164,70.3,5.7,19.27,350.0,12.78
5,6,Goa,0.7808,65,70.3,5.9,19.27,394.0,11.30
6,7,Andaman And Nicobar Islands,0.7804,36,70.3,5.8,19.27,46.0,13.37
7,8,Madhya Pradesh,0.7802,971,67.4,6.5,11.98,236.0,30.68
8,9,Bihar,0.7799,3034,69.5,5.4,7.29,1106.0,38.20
9,10,Sikkim,0.7776,41,70.3,4.1,19.27,86.0,18.58


In [7]:
# ==================== CELL 5: Export + Quick Sensitivity ====================
ranked.to_excel("Healthcare_TOPSIS_Ranked_Fixed.xlsx", index=False)
files.download("Healthcare_TOPSIS_Ranked_Fixed.xlsx")
print("✅ Downloaded: Healthcare_TOPSIS_Ranked_Fixed.xlsx (use this for report)")

# Sensitivity example
print("\n🔄 Sensitivity Example (double Disease Burden weight):")
temp_weights = ahp_weights.copy()
temp_weights[3] *= 2
temp_weights /= temp_weights.sum()
sens = topsis(df, temp_weights, benefit_cols, cost_cols)
print("New Top 5:", sens['State'].head(5).tolist())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded: Healthcare_TOPSIS_Ranked_Fixed.xlsx (use this for report)

🔄 Sensitivity Example (double Disease Burden weight):
New Top 5: ['Maharashtra', 'Karnataka', 'Tamil Nadu', 'Bihar', 'Madhya Pradesh']
